# 架构

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# 工具：把 shape 格式化成 "BxCxHxW" 形式
# ============================================================
def _fmt(shape):
    return "x".join(str(s) for s in shape)


def register_shape_hooks(model):
    """给所有叶子 module 注册 forward hook，自动打印 in/out 维度"""
    handles = []

    def make_hook(name):
        def hook(module, inputs, outputs):
            ins = [_fmt(t.shape) for t in inputs if isinstance(t, torch.Tensor)]
            if isinstance(outputs, torch.Tensor):
                outs = [_fmt(outputs.shape)]
            elif isinstance(outputs, (tuple, list)):
                outs = [_fmt(o.shape) for o in outputs if isinstance(o, torch.Tensor)]
            else:
                outs = ["?"]
            print(f"  {name:18s} | {module.__class__.__name__:8s} "
                  f"| in {ins} -> out {outs}")
        return hook

    for name, module in model.named_modules():
        if len(list(module.children())) == 0:          # 只挂在叶子层上
            handles.append(module.register_forward_hook(make_hook(name)))
    return handles


# ============================================================
# 1. Encoder: q_phi(z|x) = N(mu, diag(sigma^2))
#    784 -> 400 (ReLU) -> 400 -> {mu:20, logvar:20}
# ============================================================
class Encoder(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=20):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)   # 输出 log(sigma^2)

    def forward(self, x):
        h = F.relu(self.fc1(x))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar


# ============================================================
# 2. Decoder: p_theta(x|z)
#    20 -> 400 (ReLU) -> 400 -> 784 (Sigmoid, 伯努利均值)
# ============================================================
class Decoder(nn.Module):
    def __init__(self, latent_dim=20, hidden_dim=400, output_dim=784):
        super().__init__()
        self.fc1 = nn.Linear(latent_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, z):
        h = F.relu(self.fc1(z))
        x_hat = torch.sigmoid(self.fc2(h))
        return x_hat


# ============================================================
# 3. VAE: 组合 + Reparameterization Trick
#    z = mu + eps * sigma,  eps ~ N(0, I)
# ============================================================
class VAE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=20):
        super().__init__()
        self.encoder = Encoder(input_dim, hidden_dim, latent_dim)
        self.decoder = Decoder(latent_dim, hidden_dim, input_dim)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)          # sigma = exp(0.5 * log(sigma^2))
        eps = torch.randn_like(std)            # eps ~ N(0, I)
        return mu + eps * std

    def forward(self, x, verbose=False):
        x_flat = x.view(x.size(0), -1)         # (B,1,28,28) -> (B,784)
        mu, logvar = self.encoder(x_flat)
        z = self.reparameterize(mu, logvar)
        if verbose:
            print(f"  {'reparameterize':18s} | {'function':8s} "
                  f"| in ['mu {_fmt(mu.shape)}', 'logvar {_fmt(logvar.shape)}'] "
                  f"-> out ['z {_fmt(z.shape)}']")
        x_hat = self.decoder(z)
        return x_hat, mu, logvar


# ============================================================
# 测试：喂一个 batch，打印所有维度
# ============================================================
if __name__ == "__main__":
    torch.manual_seed(0)

    B, C, H, W = 8, 1, 28, 28                 # 模拟 MNIST batch
    x = torch.randn(B, C, H, W)

    vae = VAE(input_dim=784, hidden_dim=400, latent_dim=20)
    handles = register_shape_hooks(vae)

    print(f"input: {_fmt(x.shape)}")
    x_hat, mu, logvar = vae(x, verbose=True)
    print(f"output: x_hat {_fmt(x_hat.shape)} | mu {_fmt(mu.shape)} "
          f"| logvar {_fmt(logvar.shape)}")

    for h in handles:
        h.remove()

    # 如果想还原成图片维度：
    print("x_hat.view(B,1,28,28) ->", _fmt(x_hat.view(B, 1, 28, 28).shape))

input: 8x1x28x28
  encoder.fc1        | Linear   | in ['8x784'] -> out ['8x400']
  encoder.fc_mu      | Linear   | in ['8x400'] -> out ['8x20']
  encoder.fc_logvar  | Linear   | in ['8x400'] -> out ['8x20']
  reparameterize     | function | in ['mu 8x20', 'logvar 8x20'] -> out ['z 8x20']
  decoder.fc1        | Linear   | in ['8x20'] -> out ['8x400']
  decoder.fc2        | Linear   | in ['8x400'] -> out ['8x784']
output: x_hat 8x784 | mu 8x20 | logvar 8x20
x_hat.view(B,1,28,28) -> 8x1x28x28


----


# 示例

In [4]:
import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np


# =====================================================================
# 0. 全局配置
# =====================================================================
SEED         = 42
BATCH_SIZE   = 128
EPOCHS       = 20
LR           = 1e-3
INPUT_DIM    = 784
HIDDEN_DIM   = 400
LATENT_DIM   = 20
TRAIN_SUBSET = None      # 小数据集加速；设为 None 用完整 60000
TEST_SUBSET  = None       # 设为 None 用完整 10000
OUT_DIR      = "vae_outputs"

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")


# =====================================================================
# 1. 模型（与论文结构一致）
# =====================================================================
class Encoder(nn.Module):
    """q_phi(z|x): 784 -> 400 (ReLU) -> {mu:20, logvar:20}"""
    def __init__(self, input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM,
                 latent_dim=LATENT_DIM):
        super().__init__()
        self.fc1      = nn.Linear(input_dim, hidden_dim)
        self.fc_mu    = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x):
        h = F.relu(self.fc1(x))
        return self.fc_mu(h), self.fc_logvar(h)


class Decoder(nn.Module):
    """p_theta(x|z): 20 -> 400 (ReLU) -> 784 (Sigmoid)"""
    def __init__(self, latent_dim=LATENT_DIM, hidden_dim=HIDDEN_DIM,
                 output_dim=INPUT_DIM):
        super().__init__()
        self.fc1 = nn.Linear(latent_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, z):
        h = F.relu(self.fc1(z))
        return torch.sigmoid(self.fc2(h))


class VAE(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM,
                 latent_dim=LATENT_DIM):
        super().__init__()
        self.encoder = Encoder(input_dim, hidden_dim, latent_dim)
        self.decoder = Decoder(latent_dim, hidden_dim, input_dim)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)        # sigma = exp(0.5 * log(sigma^2))
        eps = torch.randn_like(std)          # eps ~ N(0, I)
        return mu + eps * std

    def forward(self, x):
        x = x.view(x.size(0), -1)
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar


# =====================================================================
# 2. 损失函数：ELBO 的负值（BCE + KL）
# =====================================================================
def vae_loss(recon_x, x, mu, logvar):
    x_flat = x.view(x.size(0), -1)
    bce = F.binary_cross_entropy(recon_x, x_flat, reduction="sum")
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return bce + kld, bce, kld


# =====================================================================
# 3. 数据加载
# =====================================================================
def get_loaders():
    transform = transforms.ToTensor()   # 像素值自动缩放到 [0,1]
    train_set = datasets.MNIST("./data", train=True,  download=True,
                               transform=transform)
    test_set  = datasets.MNIST("./data", train=False, download=True,
                               transform=transform)

    if TRAIN_SUBSET is not None:
        train_set = Subset(train_set, range(TRAIN_SUBSET))
    if TEST_SUBSET is not None:
        test_set = Subset(test_set, range(TEST_SUBSET))

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False)
    return train_loader, test_loader


# =====================================================================
# 4. 训练 / 评估
# =====================================================================
def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = total_bce = total_kld = 0.0
    for x, _ in loader:
        x = x.to(device)
        optimizer.zero_grad()
        recon, mu, logvar = model(x)
        loss, bce, kld = vae_loss(recon, x, mu, logvar)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_bce  += bce.item()
        total_kld  += kld.item()
    N = len(loader.dataset)
    return total_loss / N, total_bce / N, total_kld / N


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total = 0.0
    for x, _ in loader:
        x = x.to(device)
        recon, mu, logvar = model(x)
        loss, _, _ = vae_loss(recon, x, mu, logvar)
        total += loss.item()
    return total / len(loader.dataset)


# =====================================================================
# 5. 绘图
# =====================================================================
def plot_loss_curves(history, path):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(history["train_loss"], marker="o", label="train")
    ax.plot(history["test_loss"],  marker="s", label="test")
    ax.set_xlabel("epoch")
    ax.set_ylabel("negative ELBO (nats / sample)")
    ax.set_title("VAE training curve (MNIST)")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


@torch.no_grad()
def plot_reconstructions(model, loader, path, n=8):
    model.eval()
    x, _ = next(iter(loader))
    x = x[:n].to(device)
    recon, _, _ = model(x)
    recon = recon.cpu().view(-1, 28, 28)
    x = x.cpu()

    fig, axes = plt.subplots(2, n, figsize=(n * 1.4, 3.2))
    for i in range(n):
        axes[0, i].imshow(x[i, 0], cmap="gray"); axes[0, i].axis("off")
        axes[1, i].imshow(recon[i],  cmap="gray"); axes[1, i].axis("off")
    fig.subplots_adjust(left=0.10)
    fig.text(0.02, 0.72, "Original",  va="center", fontsize=11)
    fig.text(0.02, 0.28, "Recon.",    va="center", fontsize=11)
    fig.suptitle("Reconstruction on test set")
    fig.savefig(path, dpi=150)
    plt.close(fig)


@torch.no_grad()
def plot_samples(model, path, n=64, ncols=8):
    model.eval()
    z = torch.randn(n, LATENT_DIM, device=device)
    samples = model.decoder(z).cpu().view(-1, 28, 28)
    nrows = n // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols, nrows))
    for i, ax in enumerate(axes.flat):
        ax.imshow(samples[i], cmap="gray"); ax.axis("off")
    fig.suptitle("Samples from prior  z ~ N(0, I)")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


@torch.no_grad()
def plot_latent_space(model, loader, path, n=2000):
    """编码测试集得到 mu，用 SVD 做 PCA 降到 2D 可视化（按数字类别着色）"""
    model.eval()
    zs, ys = [], []
    for x, y in loader:
        x = x.to(device)
        mu, _ = model.encoder(x.view(x.size(0), -1))
        zs.append(mu.cpu()); ys.append(y)
        if sum(len(a) for a in zs) >= n:
            break
    Z = torch.cat(zs)[:n]
    Y = torch.cat(ys)[:n].numpy()

    Zc = Z - Z.mean(0, keepdim=True)
    _, _, Vh = torch.linalg.svd(Zc, full_matrices=False)
    P = (Zc @ Vh[:2].T).numpy()

    fig, ax = plt.subplots(figsize=(6, 5))
    sc = ax.scatter(P[:, 0], P[:, 1], c=Y, cmap="tab10", s=6, alpha=0.75)
    fig.colorbar(sc, ax=ax, label="digit")
    ax.set_title("Latent space (mu) projected by PCA")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


# =====================================================================
# 6. 主流程
# =====================================================================
def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    train_loader, test_loader = get_loaders()
    print(f"train samples: {len(train_loader.dataset)} | "
          f"test samples: {len(test_loader.dataset)}")

    model = VAE().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    # ---------- 维度检查（只跑一次） ----------
    print("\n[Shape check]")
    xb, _ = next(iter(train_loader))
    xb = xb.to(device)
    print(f"  input             : {tuple(xb.shape)}")
    xb_flat = xb.view(xb.size(0), -1)
    print(f"  flatten           : {tuple(xb_flat.shape)}")
    mu, logvar = model.encoder(xb_flat)
    print(f"  encoder mu        : {tuple(mu.shape)}")
    print(f"  encoder logvar    : {tuple(logvar.shape)}")
    z = model.reparameterize(mu, logvar)
    print(f"  reparameterize z  : {tuple(z.shape)}")
    recon = model.decoder(z)
    print(f"  decoder out       : {tuple(recon.shape)}\n")

    # ---------- 训练 ----------
    history = {"train_loss": [], "test_loss": []}
    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        tr_loss, tr_bce, tr_kld = train_epoch(model, train_loader, optimizer)
        te_loss = evaluate(model, test_loader)
        history["train_loss"].append(tr_loss)
        history["test_loss"].append(te_loss)
        print(f"Epoch {epoch:2d}/{EPOCHS} | {time.time()-t0:5.1f}s | "
              f"train {tr_loss:7.2f} (bce {tr_bce:6.2f} + kld {tr_kld:5.2f}) | "
              f"test {te_loss:7.2f}")

    # ---------- 保存 & 出图 ----------
    torch.save(model.state_dict(), os.path.join(OUT_DIR, "vae.pt"))
    plot_loss_curves(history, os.path.join(OUT_DIR, "loss_curve.png"))
    plot_reconstructions(model, test_loader, os.path.join(OUT_DIR, "recon.png"))
    plot_samples(model, os.path.join(OUT_DIR, "samples.png"))
    plot_latent_space(model, test_loader, os.path.join(OUT_DIR, "latent_space.png"))
    print(f"\n模型和图片已保存到 {OUT_DIR}/")


if __name__ == "__main__":
    main()

device: cpu
train samples: 60000 | test samples: 10000

[Shape check]
  input             : (128, 1, 28, 28)
  flatten           : (128, 784)
  encoder mu        : (128, 20)
  encoder logvar    : (128, 20)
  reparameterize z  : (128, 20)
  decoder out       : (128, 784)

Epoch  1/20 |  19.2s | train  163.20 (bce 147.58 + kld 15.62) | test  126.61
Epoch  2/20 |  18.7s | train  121.22 (bce  99.17 + kld 22.06) | test  115.50
Epoch  3/20 |  18.7s | train  114.28 (bce  90.48 + kld 23.80) | test  111.57
Epoch  4/20 |  18.6s | train  111.32 (bce  87.02 + kld 24.30) | test  109.59
Epoch  5/20 |  18.9s | train  109.63 (bce  85.08 + kld 24.55) | test  108.19
Epoch  6/20 |  18.1s | train  108.46 (bce  83.74 + kld 24.72) | test  107.17
Epoch  7/20 |  18.0s | train  107.58 (bce  82.75 + kld 24.83) | test  106.54
Epoch  8/20 |  17.4s | train  106.96 (bce  82.07 + kld 24.90) | test  106.12
Epoch  9/20 |  17.2s | train  106.47 (bce  81.49 + kld 24.98) | test  105.64
Epoch 10/20 |  18.1s | train  106.0